In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, cv2
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from read_images import read_images
from pre_processing import process_image_set
from pyfing import LeaderParameters, Leader, Snfoe
from pyfing.utils.drawing import draw_minutiae
from minutiae_detection import detect_minutiae_automatic

In [ ]:
# detect minutiae
def detect_minutiae(image_folder, max_block, minutia_quality_threshold):
    acquisitions = read_images(image_folder)

    for i, imgset in enumerate(acquisitions):
        imgset = process_image_set(imgset, histEq=True, target_ridge_distance=10,
                                   max_block=max_block)
        
    params = LeaderParameters(minutia_quality_threshold=minutia_quality_threshold)
    ldr = Leader(parameters=params)
    snfoe = Snfoe()

    for j, acquisition in enumerate(acquisitions):
        # Keep only the four cardinal illumination directions
        acquisition = [img for img in acquisition
                       if img.direction.lower() in ('north', 'south', 'east', 'west')]

        # Detect annotations and automatic minutiae once per image
        for img in acquisition:
            if getattr(img, 'minutiae', None) is None:
                minutiae = detect_minutiae_automatic(img, ldr)
                y0, x0 = np.asarray(img.roiCoords) * img.median_scale
                updated_minutiae = [m._replace(x=int(m.x - x0), y=int(m.y - y0)) for m in minutiae]
                img.minutiae = updated_minutiae

    return acquisitions

In [ ]:
image_folder = '../Data/Clay/CamClose/LightFar/HighAngle/Faint/Aligned/'
images = detect_minutiae(image_folder, max_block=128, minutia_quality_threshold=0.6)

In [ ]:
# save impressions with detected minutiae
save_folder = './Figures/MinutiaeVis/'
impression = 'Clay_CCLFHA_Faint'
save_path = f'{save_folder}/{impression}/'
os.makedirs(save_path, exist_ok=True)

for i, session in enumerate(images):
    for image in session:
        direction = image.direction
        minutiae = draw_minutiae(cv2.cvtColor(image.normalisedROIC, cv2.COLOR_BGR2RGB), image.minutiae, radius=5, line_length=4)
        plt.figure(figsize=(5, 3), dpi=300)
        plt.imshow(minutiae)
        plt.axis('off')
        plt.margins(x=0, y=0)
        plt.savefig(f'{save_path}/session{i+1}_direction{direction}.png', bbox_inches='tight', pad_inches=0)
        plt.close()